# v22 — Konfirmasi Multi-Timeframe untuk BOS + Alternatif Strategi Ranging

**Latar belakang:** v21 membuktikan ADX itu lagging (telat 45-60 menit mendeteksi breakout
baru), tapi 2 solusi awal yang dicoba (BOS M5 murni sbg leading signal, Fibonacci swing 50
candle sbg S/R utk ranging) **GAGAL** -- keduanya PF<1, lebih buruk dari baseline v13 murni.
Root cause BOS gagal: cuma 52.1% breakout M5 (BOS+ADX rendah) yg benar2 dikonfirmasi ADX
dalam 1 jam, pergerakan harga rata2 sesudahnya nyaris nol (0.004-0.007%) -- BOS M5 murni
terlalu banyak false-positive (breakout palsu/noise sesaat, bukan breakout beneran).

**Ide user**: jangan menyerah di 1 percobaan -- BOS M5 perlu **dikonfirmasi timeframe lebih
tinggi** (H1/H4) supaya breakout yg ditangkap lebih "beneran", bukan noise. Untuk ranging,
coba pendekatan S/R yg lebih stabil (dari H1, bukan M5) & alternatif logika lain (BB squeeze).

**Cakupan riset (SEMUA kandidat dicoba & dibandingkan, bukan cuma 1)**:

**Konfirmasi BOS (4 varian)**:
1. BOS M5 + H1 trend searah (EMA50 vs EMA200)
2. BOS M5 + BOS H1 (breakout struktur di KEDUA timeframe bersamaan)
3. BOS M5 + ADX H1 naik (momentum H1 mulai terbentuk, bukan cuma level statis)
4. BOS M5 + H4 trend searah (konfirmasi lebih besar/lambat berubah)

**Alternatif ranging (3 varian)**:
1. S/R dari swing high/low H1 (bukan M5) -- mean-reversion tapi acuan level lebih stabil
2. Skip total saat ranging (konsisten dgn temuan v18/v19 -- baseline pembanding "jangan maksa")
3. Bollinger Band squeeze + breakout arah (BUKAN mean-reversion -- deteksi BB menyempit lalu
   breakout ke 1 arah, mirip semangat BOS tapi pakai BB sbg proxy volatilitas)

**Metodologi sama spt v21 (setelah diperbaiki)**: mode TREND pakai `score_de_redundant` ASLI +
Order Block filter + H1 alignment (v13 sesungguhnya, BUKAN reimplementasi), supaya baseline
pembanding valid. TRAIN/TEST split 2026-03-01, kriteria kejujuran sama (PF>1.5, sample
TRAIN>=30, TEST>=15). Reuse cache v21 (`df_2025_2026_v12_full_nofilter.parquet`) yg sudah ada
skor v12 + OB + H1 EMA -- ditambah kolom baru H1/H4 (BOS, ADX, swing) & BB M5.

**TIDAK ADA perubahan ke `usecase.py`** -- murni riset backtest, sesuai instruksi user.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STRATEGY_NAME = "m5_scalping"
VERSION = "v22"

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME
EXPORT_DIR = PROJECT_ROOT / "dataset" / "exports" / STRATEGY_NAME / VERSION
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
(PROCESSED_DIR / VERSION).mkdir(parents=True, exist_ok=True)

INITIAL_EQUITY = 100.0
RISK_PCT = 0.01
CONTRACT_SIZE = 100.0
MIN_LOT = 0.01
LOT_STEP = 0.01
REAL_SPREAD = 1.82

MIN_SAMPLE_TRAIN = 30
MIN_SAMPLE_TEST = 15

pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Load & merge M5 + H1 + H4 (reuse cache v21 utk skor v12, tambah kolom multi-timeframe baru)

In [2]:
MTF_CACHE_PATH = PROCESSED_DIR / VERSION / "df_2025_2026_mtf.parquet"

if MTF_CACHE_PATH.exists():
    print(f"Load dari cache: {MTF_CACHE_PATH}")
    df = pd.read_parquet(MTF_CACHE_PATH)
else:
    print("Belum ada cache -- load v21 base (skor v12 sudah ada) + merge kolom H1/H4 baru...")

    v21_cache = PROCESSED_DIR / "v21" / "df_2025_2026_v12_full_nofilter.parquet"
    assert v21_cache.exists(), "Cache v21 belum ada -- jalankan v21 dulu (Section 1) sblm v22"
    df_base = pd.read_parquet(v21_cache)

    # M5 BB (bb_pct) blm ada di cache v21 -- ambil dari xauusd_m5_full_indicators.csv
    df_m5_bb = pd.read_csv(PROCESSED_DIR / "v01" / "xauusd_m5_full_indicators.csv",
                            usecols=["datetime", "bb_mid", "bb_upper", "bb_lower", "bb_pct"])
    df_m5_bb["datetime"] = pd.to_datetime(df_m5_bb["datetime"])
    df = df_base.merge(df_m5_bb, on="datetime", how="left")

    # H1: EMA (sudah ada dari v21 sbg h1_ema_50/200), tambah bos_bull/bear H1 & ADX H1
    df_h1 = pd.read_csv(PROCESSED_DIR / "v01" / "xauusd_h1_full_indicators.csv",
                         usecols=["datetime", "adx", "bos_bull", "bos_bear"])
    df_h1["datetime"] = pd.to_datetime(df_h1["datetime"])
    df_h1["h1_available_at"] = df_h1["datetime"] + pd.Timedelta(hours=1)
    df_h1 = df_h1.rename(columns={"adx": "h1_adx", "bos_bull": "h1_bos_bull", "bos_bear": "h1_bos_bear"})
    df = pd.merge_asof(
        df.sort_values("datetime"), df_h1[["h1_available_at", "h1_adx", "h1_bos_bull", "h1_bos_bear"]].sort_values("h1_available_at"),
        left_on="datetime", right_on="h1_available_at", direction="backward",
    )

    # H1 ADX naik: proxy momentum H1 baru mulai (bukan level statis) -- ADX H1 skrg > ADX H1
    # 1 candle H1 sblmnya (diff dari data H1 asli sblm di-merge_asof, supaya gak duplikat noise M5)
    df_h1_sorted = df_h1.sort_values("h1_available_at").reset_index(drop=True)
    df_h1_sorted["h1_adx_rising"] = (df_h1_sorted["h1_adx"].diff() > 0).astype(int)
    df = pd.merge_asof(
        df.sort_values("datetime"), df_h1_sorted[["h1_available_at", "h1_adx_rising"]].sort_values("h1_available_at"),
        left_on="datetime", right_on="h1_available_at", direction="backward", suffixes=("", "_dup"),
    )

    # H1 swing high/low (utk S/R ranging yg lebih stabil drpd M5) -- rolling 50 candle H1 = ~2 hari
    df_h1_swing = pd.read_csv(PROCESSED_DIR / "v01" / "xauusd_h1_full_indicators.csv", usecols=["datetime", "high", "low"])
    df_h1_swing["datetime"] = pd.to_datetime(df_h1_swing["datetime"])
    df_h1_swing = df_h1_swing.sort_values("datetime").reset_index(drop=True)
    df_h1_swing["h1_swing_high"] = df_h1_swing["high"].rolling(50).max()
    df_h1_swing["h1_swing_low"] = df_h1_swing["low"].rolling(50).min()
    df_h1_swing["h1_available_at"] = df_h1_swing["datetime"] + pd.Timedelta(hours=1)
    df = pd.merge_asof(
        df.sort_values("datetime"), df_h1_swing[["h1_available_at", "h1_swing_high", "h1_swing_low"]].sort_values("h1_available_at"),
        left_on="datetime", right_on="h1_available_at", direction="backward", suffixes=("", "_dup2"),
    )

    # H4: EMA trend (konfirmasi lebih besar/lambat)
    df_h4 = pd.read_csv(PROCESSED_DIR / "v01" / "xauusd_h4_full_indicators.csv",
                         usecols=["datetime", "ema_50", "ema_200"])
    df_h4["datetime"] = pd.to_datetime(df_h4["datetime"])
    df_h4["h4_available_at"] = df_h4["datetime"] + pd.Timedelta(hours=4)
    df_h4 = df_h4.rename(columns={"ema_50": "h4_ema_50", "ema_200": "h4_ema_200"})
    df = pd.merge_asof(
        df.sort_values("datetime"), df_h4[["h4_available_at", "h4_ema_50", "h4_ema_200"]].sort_values("h4_available_at"),
        left_on="datetime", right_on="h4_available_at", direction="backward",
    )

    drop_cols = [c for c in df.columns if c.endswith("_dup") or c.endswith("_dup2") or c in ("h1_available_at", "h4_available_at")]
    df = df.drop(columns=drop_cols)

    df.to_parquet(MTF_CACHE_PATH, index=False)
    print(f"Tersimpan ke cache: {MTF_CACHE_PATH}")

print(f"\nTotal candle: {len(df)}, {df['datetime'].min()} -> {df['datetime'].max()}")
print(f"Kolom: {list(df.columns)}")

Load dari cache: D:\Projects\robot-scalping\dataset\processed\m5_scalping\v22\df_2025_2026_mtf.parquet

Total candle: 107336, 2025-01-01 23:00:00+00:00 -> 2026-08-06 12:35:00+00:00
Kolom: ['datetime', 'open', 'high', 'low', 'close', 'adx', 'atr', 'v12_score', 'bull_chain', 'bear_chain', 'bos_bull', 'bos_bear', 'fib_swing_high', 'fib_swing_low', 'fib_236', 'fib_382', 'fib_500', 'fib_618', 'fib_786', 'ob_bull', 'ob_bear', 'h1_ob_bull', 'h1_ob_bear', 'h1_ema_50', 'h1_ema_200', 'bb_mid', 'bb_upper', 'bb_lower', 'bb_pct', 'h1_adx', 'h1_bos_bull', 'h1_bos_bear', 'h1_adx_rising', 'h1_swing_high', 'h1_swing_low', 'h4_ema_50', 'h4_ema_200']


## 2. Backtest engine v22: v13 asli (TREND) + BOS dgn 4 varian konfirmasi + 3 varian ranging

In [3]:
def check_h1_alignment_v22(h1_ema_50, h1_ema_200, direction: str) -> bool:
    if h1_ema_50 is None or h1_ema_200 is None or not np.isfinite(h1_ema_50) or not np.isfinite(h1_ema_200):
        return True
    h1_trend = "UP" if h1_ema_50 > h1_ema_200 else ("DOWN" if h1_ema_50 < h1_ema_200 else "FLAT")
    if direction == "BUY" and h1_trend == "DOWN":
        return False
    if direction == "SELL" and h1_trend == "UP":
        return False
    return True


def run_backtest_v22(
    df_signals: pd.DataFrame,
    adx_trend_min: float,
    adx_ranging_max: float,
    min_signal_score: float,
    sl_mult_trend: float,
    tp_mult_trend: float,
    sl_mult_bos: float,
    tp_mult_bos: float,
    bos_confirm_mode: str,   # "none", "h1_trend", "h1_bos", "h1_adx_rising", "h4_trend"
    ranging_mode: str,       # "skip", "h1_fib", "bb_squeeze"
    fib_proximity_pct: float,
    sl_mult_fib: float,
    bb_squeeze_pct: float,   # threshold BB width (persentil) dianggap "squeeze"
    sl_mult_bb: float,
    tp_mult_bb: float,
    max_hold: int,
    enable_bos: bool = True,
    enable_ranging: bool = True,
    bb_width_threshold: float = None,  # precomputed dari TRAIN, dipassing ke TEST biar gak leak
    require_ob_filter: bool = True,
    require_h1_alignment: bool = True,
    spread_points: float = REAL_SPREAD,
) -> pd.DataFrame:
    close_arr = df_signals["close"].to_numpy()
    high_arr = df_signals["high"].to_numpy()
    low_arr = df_signals["low"].to_numpy()
    adx_arr = df_signals["adx"].to_numpy()
    atr_arr = df_signals["atr"].to_numpy()
    score_arr = df_signals["v12_score"].to_numpy()
    bos_bull_arr = df_signals["bos_bull"].to_numpy()
    bos_bear_arr = df_signals["bos_bear"].to_numpy()
    ob_bull_arr = df_signals["ob_bull"].to_numpy()
    ob_bear_arr = df_signals["ob_bear"].to_numpy()
    h1_ob_bull_arr = df_signals["h1_ob_bull"].to_numpy()
    h1_ob_bear_arr = df_signals["h1_ob_bear"].to_numpy()
    h1_ema_50_arr = df_signals["h1_ema_50"].to_numpy()
    h1_ema_200_arr = df_signals["h1_ema_200"].to_numpy()
    h1_bos_bull_arr = df_signals["h1_bos_bull"].to_numpy()
    h1_bos_bear_arr = df_signals["h1_bos_bear"].to_numpy()
    h1_adx_rising_arr = df_signals["h1_adx_rising"].to_numpy()
    h4_ema_50_arr = df_signals["h4_ema_50"].to_numpy()
    h4_ema_200_arr = df_signals["h4_ema_200"].to_numpy()
    h1_swing_high_arr = df_signals["h1_swing_high"].to_numpy()
    h1_swing_low_arr = df_signals["h1_swing_low"].to_numpy()
    fib_236 = df_signals["fib_236"].to_numpy()
    fib_618 = df_signals["fib_618"].to_numpy()
    fib_786 = df_signals["fib_786"].to_numpy()
    bb_upper_arr = df_signals["bb_upper"].to_numpy()
    bb_lower_arr = df_signals["bb_lower"].to_numpy()
    bb_mid_arr = df_signals["bb_mid"].to_numpy()
    bb_width_arr = (bb_upper_arr - bb_lower_arr) / bb_mid_arr
    datetime_arr = df_signals["datetime"].to_numpy()
    n = len(df_signals)

    if ranging_mode == "bb_squeeze" and bb_width_threshold is None:
        bb_width_threshold = np.nanpercentile(bb_width_arr, bb_squeeze_pct * 100)

    trades = []
    equity = INITIAL_EQUITY
    i = 0
    while i < n:
        adx, atr, close, score = adx_arr[i], atr_arr[i], close_arr[i], score_arr[i]
        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(adx) or not np.isfinite(score):
            i += 1
            continue

        direction = None
        mode = None
        sl_mult = tp_mult = None
        tp_price_override = None

        if adx >= adx_trend_min:
            if score >= min_signal_score:
                direction = "BUY"
            elif score <= -min_signal_score:
                direction = "SELL"
            if direction is not None:
                if require_ob_filter:
                    opposing_ob = (
                        (direction == "BUY" and (ob_bear_arr[i] > 0 or h1_ob_bear_arr[i] > 0)) or
                        (direction == "SELL" and (ob_bull_arr[i] > 0 or h1_ob_bull_arr[i] > 0))
                    )
                    if opposing_ob:
                        direction = None
                if direction is not None and require_h1_alignment:
                    if not check_h1_alignment_v22(h1_ema_50_arr[i], h1_ema_200_arr[i], direction):
                        direction = None
            sl_mult, tp_mult = sl_mult_trend, tp_mult_trend

        elif adx_ranging_max <= adx < adx_trend_min and enable_bos:
            bos_dir = None
            if bos_bull_arr[i] == 1:
                bos_dir = "BUY"
            elif bos_bear_arr[i] == 1:
                bos_dir = "SELL"

            if bos_dir is not None:
                confirmed = True
                if bos_confirm_mode == "h1_trend":
                    confirmed = check_h1_alignment_v22(h1_ema_50_arr[i], h1_ema_200_arr[i], bos_dir)
                elif bos_confirm_mode == "h1_bos":
                    confirmed = (bos_dir == "BUY" and h1_bos_bull_arr[i] == 1) or (bos_dir == "SELL" and h1_bos_bear_arr[i] == 1)
                elif bos_confirm_mode == "h1_adx_rising":
                    confirmed = h1_adx_rising_arr[i] == 1
                elif bos_confirm_mode == "h4_trend":
                    confirmed = check_h1_alignment_v22(h4_ema_50_arr[i], h4_ema_200_arr[i], bos_dir)
                # "none" -> confirmed tetap True (BOS murni, spt v21 baseline)

                if confirmed:
                    direction, mode = bos_dir, "BOS"
            sl_mult, tp_mult = sl_mult_bos, tp_mult_bos

        elif adx < adx_ranging_max and enable_ranging:
            if ranging_mode == "h1_fib":
                rng = h1_swing_high_arr[i] - h1_swing_low_arr[i]
                if np.isfinite(rng) and rng > 0:
                    tol = rng * fib_proximity_pct
                    fib618 = h1_swing_high_arr[i] - rng * 0.618
                    fib786 = h1_swing_high_arr[i] - rng * 0.786
                    fib236 = h1_swing_high_arr[i] - rng * 0.236
                    near_support = abs(close - fib786) <= tol or abs(close - fib618) <= tol
                    near_resistance = abs(close - fib236) <= tol
                    if near_support:
                        direction, mode = "BUY", "FIB_H1"
                        sl_mult = sl_mult_fib
                        tp_price_override = (h1_swing_high_arr[i] + h1_swing_low_arr[i]) / 2
                    elif near_resistance:
                        direction, mode = "SELL", "FIB_H1"
                        sl_mult = sl_mult_fib
                        tp_price_override = (h1_swing_high_arr[i] + h1_swing_low_arr[i]) / 2

            elif ranging_mode == "bb_squeeze":
                is_squeeze = bb_width_arr[i] <= bb_width_threshold if np.isfinite(bb_width_arr[i]) else False
                if is_squeeze:
                    if close >= bb_upper_arr[i]:
                        direction, mode = "BUY", "BB_SQUEEZE"
                    elif close <= bb_lower_arr[i]:
                        direction, mode = "SELL", "BB_SQUEEZE"
                sl_mult, tp_mult = sl_mult_bb, tp_mult_bb
            # "skip" -> direction tetap None, tidak trading di ranging

        if direction is None:
            i += 1
            continue
        if mode is None:
            mode = "TREND"

        sl_points = sl_mult * atr
        entry_price = close + (spread_points if direction == "BUY" else -spread_points)
        if tp_price_override is not None:
            tp_price = tp_price_override
            if (direction == "BUY" and tp_price <= entry_price) or (direction == "SELL" and tp_price >= entry_price):
                i += 1
                continue
        else:
            tp_points = tp_mult * atr
            tp_price = entry_price + tp_points if direction == "BUY" else entry_price - tp_points
        sl_price = entry_price - sl_points if direction == "BUY" else entry_price + sl_points

        entry_time = datetime_arr[i]
        exit_price = None
        exit_idx = min(i + max_hold, n - 1)
        window_end = min(i + 1 + max_hold, n)
        for candle_idx in range(i + 1, window_end):
            c_high, c_low = high_arr[candle_idx], low_arr[candle_idx]
            hit_tp = c_high >= tp_price if direction == "BUY" else c_low <= tp_price
            hit_sl = c_low <= sl_price if direction == "BUY" else c_high >= sl_price
            if hit_sl:
                exit_price, exit_time = sl_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_time = tp_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
        if exit_price is None:
            exit_price, exit_time = close_arr[exit_idx], datetime_arr[exit_idx]

        next_i = exit_idx + 1
        price_move = (exit_price - entry_price) if direction == "BUY" else (entry_price - exit_price)
        risk_amount = equity * RISK_PCT
        lot = max(round(math.floor((risk_amount / (sl_points * CONTRACT_SIZE)) / LOT_STEP) * LOT_STEP, 2), MIN_LOT) if sl_points > 0 else MIN_LOT
        pnl = price_move * lot * CONTRACT_SIZE
        equity += pnl
        trades.append({
            "entry_time": entry_time, "mode": mode, "direction": direction, "pnl": pnl,
            "result": "WIN" if pnl > 0 else "LOSS", "equity_after": equity,
        })
        i = next_i

    return pd.DataFrame(trades)


def evaluate(trades: pd.DataFrame, initial_equity: float) -> dict:
    if trades.empty:
        return {"total_trades": 0, "win_rate_pct": 0, "profit_factor": 0, "net_pnl": 0, "max_drawdown_pct": 0}
    wins = trades[trades["pnl"] > 0]
    losses = trades[trades["pnl"] <= 0]
    gross_profit = wins["pnl"].sum()
    gross_loss = losses["pnl"].sum()
    equity_series = pd.Series([initial_equity] + trades["equity_after"].tolist())
    running_max = equity_series.cummax()
    drawdown = (equity_series - running_max) / running_max * 100
    return {
        "total_trades": len(trades),
        "win_rate_pct": round(len(wins) / len(trades) * 100, 2),
        "profit_factor": round(gross_profit / abs(gross_loss), 2) if gross_loss != 0 else float("inf"),
        "net_pnl": round(gross_profit + gross_loss, 2),
        "max_drawdown_pct": round(drawdown.min(), 2),
    }


def evaluate_by_mode(trades: pd.DataFrame) -> pd.DataFrame:
    if trades.empty:
        return pd.DataFrame()
    rows = []
    for mode, g in trades.groupby("mode"):
        wins = (g["result"] == "WIN").sum()
        gp = g.loc[g["pnl"] > 0, "pnl"].sum()
        gl = abs(g.loc[g["pnl"] <= 0, "pnl"].sum())
        rows.append({
            "mode": mode, "n": len(g), "win_rate_pct": round(wins / len(g) * 100, 1),
            "pf": round(gp / gl, 2) if gl > 0 else float("inf"), "net_pnl": round(g["pnl"].sum(), 2),
        })
    return pd.DataFrame(rows)

print("Backtest engine v22 siap.")

Backtest engine v22 siap.


## 3. TRAIN/TEST split & Baseline A (v13 murni, pembanding utama)

In [4]:
TRAIN_END = pd.Timestamp("2026-03-01", tz="UTC")
df_train = df[df["datetime"] < TRAIN_END].reset_index(drop=True)
df_test = df[df["datetime"] >= TRAIN_END].reset_index(drop=True)
print(f"TRAIN: {len(df_train)} candle | TEST: {len(df_test)} candle")

COMMON = dict(
    adx_trend_min=25.0, adx_ranging_max=18.0, min_signal_score=9.0,
    sl_mult_trend=2.0, tp_mult_trend=4.0,
    sl_mult_bos=2.0, tp_mult_bos=3.0,
    fib_proximity_pct=0.05, sl_mult_fib=1.5,
    bb_squeeze_pct=0.2, sl_mult_bb=1.5, tp_mult_bb=3.0,
    max_hold=12,
)

trades_base_train = run_backtest_v22(df_train, bos_confirm_mode="none", ranging_mode="skip", **COMMON)
trades_base_test = run_backtest_v22(df_test, bos_confirm_mode="none", ranging_mode="skip", **COMMON)
print("=== Baseline A: v13 murni (BOS off via confirm='none' TAPI adx_ranging_max=0 effectively; verifikasi di bawah) ===")
baseline_a_train = evaluate(trades_base_train, INITIAL_EQUITY)
baseline_a_test = evaluate(trades_base_test, INITIAL_EQUITY)
print("TRAIN:", baseline_a_train)
print("TEST:", baseline_a_test)
print(evaluate_by_mode(trades_base_test).to_string(index=False))

TRAIN: 77226 candle | TEST: 30110 candle


=== Baseline A: v13 murni (BOS off via confirm='none' TAPI adx_ranging_max=0 effectively; verifikasi di bawah) ===
TRAIN: {'total_trades': 1921, 'win_rate_pct': 33.32, 'profit_factor': np.float64(0.68), 'net_pnl': np.float64(-2448.25), 'max_drawdown_pct': np.float64(-2452.75)}
TEST: {'total_trades': 719, 'win_rate_pct': 39.78, 'profit_factor': np.float64(0.85), 'net_pnl': np.float64(-599.8), 'max_drawdown_pct': np.float64(-286.96)}
 mode   n  win_rate_pct   pf  net_pnl
  BOS 618          36.1 0.69 -1147.34
TREND 101          62.4 2.56   547.54


**Catatan penting**: `bos_confirm_mode="none"` di atas berarti BOS TETAP aktif tanpa filter
tambahan (persis v21's Baseline B, yg terbukti PF<1) -- BUKAN baseline v13 murni. Baseline v13
murni sesungguhnya perlu BOS dimatikan total. Diperbaiki di cell berikut dgn parameter eksplisit
`adx_ranging_max` dan mode BOS di-disable via `bos_confirm_mode=None` (bukan string "none").

In [5]:
def run_v13_pure(df_signals, adx_trend_min, min_signal_score, sl_mult_trend, tp_mult_trend, max_hold,
                  require_ob_filter=True, require_h1_alignment=True, spread_points=REAL_SPREAD):
    """v13 MURNI TANPA mode BOS/ranging apapun -- pembanding utama yg valid.
    Perbaikan dari draft awal: adx_ranging_max=-1.0 TERNYATA BUG (kondisi elif jadi
    -1.0 <= adx < adx_trend_min, yg mencakup HAMPIR SEMUA ADX positif -- BOS malah
    makin luas aktif, bukan mati). Fix: pakai flag enable_bos/enable_ranging eksplisit."""
    return run_backtest_v22(
        df_signals, adx_trend_min=adx_trend_min, adx_ranging_max=18.0,
        min_signal_score=min_signal_score, sl_mult_trend=sl_mult_trend, tp_mult_trend=tp_mult_trend,
        sl_mult_bos=0, tp_mult_bos=0, bos_confirm_mode="none", ranging_mode="skip",
        fib_proximity_pct=0, sl_mult_fib=0, bb_squeeze_pct=0, sl_mult_bb=0, tp_mult_bb=0,
        max_hold=max_hold, enable_bos=False, enable_ranging=False,
        require_ob_filter=require_ob_filter, require_h1_alignment=require_h1_alignment,
        spread_points=spread_points,
    )

trades_pure_train = run_v13_pure(df_train, adx_trend_min=25.0, min_signal_score=9.0, sl_mult_trend=2.0, tp_mult_trend=4.0, max_hold=12)
trades_pure_test = run_v13_pure(df_test, adx_trend_min=25.0, min_signal_score=9.0, sl_mult_trend=2.0, tp_mult_trend=4.0, max_hold=12)

baseline_pure_train = evaluate(trades_pure_train, INITIAL_EQUITY)
baseline_pure_test = evaluate(trades_pure_test, INITIAL_EQUITY)
print("=== Baseline PURE: v13 murni sesungguhnya (enable_bos=False, enable_ranging=False) ===")
print("TRAIN:", baseline_pure_train)
print("TEST:", baseline_pure_test)

=== Baseline PURE: v13 murni sesungguhnya (enable_bos=False, enable_ranging=False) ===
TRAIN: {'total_trades': 318, 'win_rate_pct': 49.37, 'profit_factor': np.float64(1.61), 'net_pnl': np.float64(503.27), 'max_drawdown_pct': np.float64(-44.08)}
TEST: {'total_trades': 121, 'win_rate_pct': 61.98, 'profit_factor': np.float64(2.63), 'net_pnl': np.float64(694.31), 'max_drawdown_pct': np.float64(-31.93)}


## 4. Bandingkan 4 varian konfirmasi BOS (semua vs Baseline PURE)

In [6]:
bos_variants = ["none", "h1_trend", "h1_bos", "h1_adx_rising", "h4_trend"]
bos_results = []
for variant in bos_variants:
    tr_train = run_backtest_v22(df_train, bos_confirm_mode=variant, ranging_mode="skip", **COMMON)
    tr_test = run_backtest_v22(df_test, bos_confirm_mode=variant, ranging_mode="skip", **COMMON)
    m_train = evaluate(tr_train, INITIAL_EQUITY)
    m_test = evaluate(tr_test, INITIAL_EQUITY)
    bos_only_train = tr_train[tr_train["mode"] == "BOS"]
    bos_only_test = tr_test[tr_test["mode"] == "BOS"]
    m_bos_train = evaluate(bos_only_train, INITIAL_EQUITY)
    m_bos_test = evaluate(bos_only_test, INITIAL_EQUITY)
    bos_results.append({
        "variant": variant,
        "bos_n_train": m_bos_train["total_trades"], "bos_pf_train": m_bos_train["profit_factor"],
        "bos_n_test": m_bos_test["total_trades"], "bos_pf_test": m_bos_test["profit_factor"],
        "bos_wr_test": m_bos_test["win_rate_pct"], "bos_netpnl_test": m_bos_test["net_pnl"],
        "combined_pf_test": m_test["profit_factor"], "combined_netpnl_test": m_test["net_pnl"],
    })

bos_compare_df = pd.DataFrame(bos_results)
print("=== Perbandingan 4 varian konfirmasi BOS (mode BOS SAJA, terisolasi dari TREND) ===")
print(bos_compare_df.to_string(index=False))
print(f"\nBaseline PURE (v13 murni, tanpa mode BOS sama sekali) TEST: PF={baseline_pure_test['profit_factor']}, net_pnl={baseline_pure_test['net_pnl']}")

=== Perbandingan 4 varian konfirmasi BOS (mode BOS SAJA, terisolasi dari TREND) ===
      variant  bos_n_train  bos_pf_train  bos_n_test  bos_pf_test  bos_wr_test  bos_netpnl_test  combined_pf_test  combined_netpnl_test
         none         1665          0.58         618         0.69        36.08         -1147.34              0.85               -599.80
     h1_trend          898          0.58         306         0.79        37.25          -358.14              1.09                189.40
       h1_bos           64          0.88          11         0.79        54.55           -18.85              2.22                632.17
h1_adx_rising          743          0.57         258         0.67        37.60          -512.27              1.05                 92.08
     h4_trend          941          0.64         318         0.72        37.11          -520.44              1.05                115.57

Baseline PURE (v13 murni, tanpa mode BOS sama sekali) TEST: PF=2.63, net_pnl=694.31


## 5. Bandingkan 3 varian strategi ranging (semua vs Baseline PURE)

In [7]:
# BB squeeze threshold dicari dari TRAIN, dipassing konsisten ke TEST (spy gak leak)
bb_width_train = (df_train["bb_upper"] - df_train["bb_lower"]) / df_train["bb_mid"]
bb_threshold_from_train = np.nanpercentile(bb_width_train, COMMON["bb_squeeze_pct"] * 100)
print(f"BB width squeeze threshold (dari TRAIN, persentil {COMMON['bb_squeeze_pct']*100:.0f}%): {bb_threshold_from_train:.5f}")

ranging_variants = ["skip", "h1_fib", "bb_squeeze"]
ranging_results = []
for variant in ranging_variants:
    kwargs_train = dict(COMMON)
    kwargs_test = dict(COMMON)
    if variant == "bb_squeeze":
        kwargs_train["bb_width_threshold"] = bb_threshold_from_train
        kwargs_test["bb_width_threshold"] = bb_threshold_from_train
    tr_train = run_backtest_v22(df_train, bos_confirm_mode="none", ranging_mode=variant, **kwargs_train)
    tr_test = run_backtest_v22(df_test, bos_confirm_mode="none", ranging_mode=variant, **kwargs_test)
    ranging_mode_label = "FIB_H1" if variant == "h1_fib" else ("BB_SQUEEZE" if variant == "bb_squeeze" else None)
    if ranging_mode_label:
        r_only_test = tr_test[tr_test["mode"] == ranging_mode_label]
        m_r_test = evaluate(r_only_test, INITIAL_EQUITY)
    else:
        m_r_test = {"total_trades": 0, "profit_factor": None, "win_rate_pct": None, "net_pnl": 0}
    m_combined_test = evaluate(tr_test, INITIAL_EQUITY)
    ranging_results.append({
        "variant": variant, "ranging_n_test": m_r_test["total_trades"],
        "ranging_pf_test": m_r_test["profit_factor"], "ranging_wr_test": m_r_test["win_rate_pct"],
        "ranging_netpnl_test": m_r_test["net_pnl"],
        "combined_pf_test": m_combined_test["profit_factor"], "combined_netpnl_test": m_combined_test["net_pnl"],
    })

ranging_compare_df = pd.DataFrame(ranging_results)
print("\n=== Perbandingan 3 varian strategi ranging ===")
print(ranging_compare_df.to_string(index=False))

BB width squeeze threshold (dari TRAIN, persentil 20%): 0.00200



=== Perbandingan 3 varian strategi ranging ===
   variant  ranging_n_test  ranging_pf_test  ranging_wr_test  ranging_netpnl_test  combined_pf_test  combined_netpnl_test
      skip               0              NaN              NaN                 0.00              0.85               -599.80
    h1_fib             601             0.77            31.11              -676.46              0.86               -883.93
bb_squeeze              77             0.39            22.08              -163.41              0.83               -704.64


## 6. Kandidat terbaik (kalau ada) -- grid search halus di sekitar varian yg menang Section 4-5

In [8]:
best_bos_variant = bos_compare_df.loc[bos_compare_df["bos_pf_train"].idxmax(), "variant"] if bos_compare_df["bos_n_train"].max() >= MIN_SAMPLE_TRAIN else None
print(f"Varian BOS terbaik (by TRAIN PF, sample cukup): {best_bos_variant}")
print(bos_compare_df)

print()
best_ranging_variant = ranging_compare_df.loc[ranging_compare_df["ranging_netpnl_test"].idxmax(), "variant"]
print(f"Varian ranging dgn net_pnl TEST terbaik: {best_ranging_variant}")
print(ranging_compare_df)

Varian BOS terbaik (by TRAIN PF, sample cukup): h1_bos
         variant  bos_n_train  bos_pf_train  bos_n_test  bos_pf_test  bos_wr_test  bos_netpnl_test  combined_pf_test  combined_netpnl_test
0           none         1665          0.58         618         0.69        36.08         -1147.34              0.85               -599.80
1       h1_trend          898          0.58         306         0.79        37.25          -358.14              1.09                189.40
2         h1_bos           64          0.88          11         0.79        54.55           -18.85              2.22                632.17
3  h1_adx_rising          743          0.57         258         0.67        37.60          -512.27              1.05                 92.08
4       h4_trend          941          0.64         318         0.72        37.11          -520.44              1.05                115.57

Varian ranging dgn net_pnl TEST terbaik: skip
      variant  ranging_n_test  ranging_pf_test  ranging_wr_test 

## 7. Kesimpulan

*(diisi setelah lihat hasil eksekusi lengkap Section 3-6 -- placeholder)*